# Baseline Models for Daily Rat Sightings in Manhattan

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

from pandas.tseries.holiday import USFederalHolidayCalendar

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)


# Modeling Daily Rat Sightings in Manhattan

## Importing the Data

In [2]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2025-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to MANHATTAN

rs = rs[rs['borough']=='MANHATTAN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

rs

,created_date,count
2,2020-01-01,4
7,2020-01-02,7
12,2020-01-03,16
17,2020-01-04,10
21,2020-01-05,5
...,...,...
8905,2025-02-24,19
8910,2025-02-25,20
8915,2025-02-26,15
8920,2025-02-27,16


In [4]:
## We find the missing row and add it in.
## There's probably a better way to find in the missing dates and update it. This is a sort of hacky solution.

date_range = pd.date_range(start=rs['created_date'].min(), end=rs['created_date'].max(), freq='D')
complete_dates_df = pd.DataFrame(date_range, columns=['created_date'])
rs = pd.merge(complete_dates_df, rs, on='created_date', how='left')
rs['count'] = rs['count'].fillna(0).astype(int)
rs = rs.sort_values(by='created_date').reset_index(drop=True)

rs

,created_date,count
0,2020-01-01,4
1,2020-01-02,7
2,2020-01-03,16
3,2020-01-04,10
4,2020-01-05,5
...,...,...
1881,2025-02-24,19
1882,2025-02-25,20
1883,2025-02-26,15
1884,2025-02-27,16


## Baseline Seasonal Average Model

In [5]:
years_back_use = 4
day_window_use = 4

In [6]:
def seasonal_average_forecast(data, target_dates, years_back=years_back_use, day_window=day_window_use):
    df = data.copy()
    # ensure datetime type
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["doy"] = df["created_date"].dt.dayofyear
    df["year"] = df["created_date"].dt.year

    forecasts = []
    for target_date in target_dates:
        target_doy = target_date.dayofyear
        target_year = target_date.year
        mask = ((df["year"] >= target_year - years_back) & (df["year"] < target_year) & (np.abs(df["doy"] - target_doy) <= day_window))
        forecasts.append(df.loc[mask, "count"].mean())
    return pd.Series(forecasts, index=target_dates)

In [7]:
results = []

rs["created_date"] = pd.to_datetime(rs["created_date"])

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()
    
    # Target dates = the dates we want to forecast. There are 14 days.
    target_dates = test["created_date"]
    
    # Seasonal forecast using only the training data (we will go back 5 years and take the average and use a day_window of 5 as well.)
    y_pred = seasonal_average_forecast(data=train, target_dates=target_dates, years_back=years_back_use,day_window=day_window_use)

    # We take the true values.
    y_true = test["count"].values
    
    # Compute the metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append the results of the metrics to the table as well as the fold number.
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert the data to a table for readability.
baseline_results_df = pd.DataFrame(results)

# We also include a new row which consists of the average RMSE and MAPE over each fold.
baseline_results_df.loc["mean"] = ["mean", baseline_results_df["rmse"].mean(), baseline_results_df["mape"].mean()]

baseline_results_df

,fold,rmse,mape
0,0,4.002548,0.280540
1,1,3.973568,0.238817
2,2,5.476119,0.543911
3,3,5.317748,0.319481
4,4,5.362828,0.521556
5,5,6.716653,0.400610
6,6,6.795952,0.362424
7,7,5.487635,0.328791
8,8,6.554118,0.492396
9,9,5.527818,0.276961


## Year Ago Rolling 4 Week Average 

In [8]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Just saving a copy for later
rs_saved = rs.copy()

In [9]:
# Tired of writing np.sqrt or typing a long name.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = []

for fold, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    # Calculate the 4-week rolling average for the training data
    train_sorted = train.sort_values('ds') # making sure to sort it by date
    train_sorted['rolling_4w'] = train_sorted['y'].rolling(window=4, min_periods=1).mean()

    # This part of the code makes the predictions. We use the 'rolling_4w' column of the training set.
    y_pred = []
    y_true = test['y'].values

    for idx, row in test.iterrows():
        # Predict using the latest rolling average from the train data
        prediction = train_sorted['rolling_4w'].iloc[-1]  # Last value in the train rolling avg
        y_pred.append(prediction)
        
    # Calculate RMSE and MAPE for this fold
    fold_rmse = rmse(y_true, y_pred)
    fold_mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': fold, 'rmse': fold_rmse, 'mape': fold_mape})

rolling4w_results_df = pd.DataFrame(results)

# Optional: add a row for the overall average RMSE and MAPE
overall_rmse = rolling4w_results_df['rmse'].mean()
overall_mape = rolling4w_results_df['mape'].mean()
rolling4w_results_df.loc['mean'] = ['mean', overall_rmse, overall_mape]

In [10]:
rolling4w_results_df

,fold,rmse,mape
0,0,3.615443,0.240564
1,1,4.228939,0.291261
2,2,5.489438,0.553228
3,3,7.294200,0.306979
4,4,6.184658,0.560685
5,5,7.215435,0.457943
6,6,7.551608,0.438976
7,7,5.195293,0.290578
8,8,5.712611,0.416645
9,9,6.058495,0.281693


## Results of the Tw Baselines

In [11]:
models = {
    'baseline': baseline_results_df,
    'rolling4w': rolling4w_results_df,
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)
# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                mape          
model  baseline rolling4w  baseline rolling4w
fold                                         
0      4.002548  3.615443  0.280540  0.240564
1      3.973568  4.228939  0.238817  0.291261
2      5.476119  5.489438  0.543911  0.553228
3      5.317748  7.294200  0.319481  0.306979
4      5.362828  6.184658  0.521556  0.560685
5      6.716653  7.215435  0.400610  0.457943
6      6.795952  7.551608  0.362424  0.438976
7      5.487635  5.195293  0.328791  0.290578
8      6.554118  5.712611  0.492396  0.416645
9      5.527818  6.058495  0.276961  0.281693
10     5.623782  5.354404  0.192254  0.196827
11     6.075183  5.087625  0.339792  0.270098
12     5.326969  5.291503  0.242283  0.235474
13     4.265165  5.021383  0.214755  0.265180
14     6.462533  6.152090  0.382883  0.320797
15     6.311571  7.047669  0.328794  0.406757
16     5.309839  5.656065  0.334273  0.379940
17     5.684011  6.640649  0.424729  0.516944
18     3.431419  2.297126  0.218789  0.138973
19     4.305684  5.246598  0.443678  0.569316
20     4.474926  3.977616  1.132338  0.942156
21     4.060794  5.548488  0.733533  1.016573
22     4.660300  3.384576  0.391966  0.269467
23     7.468519  7.208006  0.565420  0.540570
24     4.362053  8.415016  0.449557  0.922839
25     5.008095  5.315073  0.552642  0.582466
mean   5.309455  5.622693  0.412045  0.438959